# 02 - Carga y filtrado del Anuario de Aforos

Este notebook carga los datos del Anuario de Aforos del MITECO (CEDEX) y los filtra por la Demarcación Hidrográfica del Miño-Sil.

## Proceso

1. Inspección inicial del alcance geográfico de los CSV (nacional vs. demarcación específica).
2. Identificación de embalses y estaciones dentro de la Demarcación Miño-Sil.
3. Filtrado de los datos diarios de embalses (AFLIQE) y de aforo en río (AFLIQ).
4. Filtrado por el periodo de estudio (2000-2023).
5. Guardado en `data/interim/` para uso posterior.

## Fuente

Anuario de Aforos - Sistema de Información CEDEX  
https://www.miteco.gob.es/es/cartografia-y-sig/ide/descargas/agua/anuario-de-aforos.html

## 1. Imports y configuración

In [1]:
from pathlib import Path
import pandas as pd

# Rutas
DIR_RAW_ANUARIO = Path("../data/raw/anuario")
DIR_INTERIM = Path("../data/interim")
DIR_INTERIM.mkdir(parents=True, exist_ok=True)

# Periodo de estudio
FECHA_INICIO = pd.Timestamp("2000-01-01")
FECHA_FIN = pd.Timestamp("2023-12-31")

# Diccionario de tipos de dato de embalse (reemplaza el fichero que falta)
TIPO_DATO_EMB = {1: "RESERVA (HM3)", 2: "SALIDA (HM3)", 3: "ENTRADA (HM3)"}

# Diccionario de identificador de gran cuenca
# El Miño-Sil es la gran cuenca 10 según el sistema del CEDEX
GR_CUENCA_MINOSIL = 14

## 2. Inspección del alcance geográfico

Para determinar si los datos son nacionales o solo de Miño-Sil, revisamos las tablas de catálogos (embalses, estaciones de aforo, gran cuenca).

In [2]:
# Cargar catálogo de grandes cuencas
gr_cuenca = pd.read_csv(DIR_RAW_ANUARIO / "gr_cuenca.csv", sep=";", encoding="latin-1")
print("=== Grandes cuencas disponibles ===")
print(gr_cuenca)

=== Grandes cuencas disponibles ===
   gr_cuenca_id                       gr_cuenca
0            18  MIÑO-SIL                      


In [3]:
# Cargar catálogo de embalses y ver de qué cuencas son
embalses = pd.read_csv(DIR_RAW_ANUARIO / "embalse.csv", sep=";", encoding="latin-1")
print(f"Total embalses en el CSV: {len(embalses)}")
print(f"\nColumnas: {list(embalses.columns)}")
print(f"\nMuestra:")
embalses.head()

Total embalses en el CSV: 35

Columnas: ['ref_ceh', 'nom_embalse', 'serv', 'comentario', 'muni_id', 'hoja_id', 'xutm', 'yutm', 'xutm30', 'yutm30', 'long', 'lat', 'longwgs84', 'latwgs84', 'mna', 'mnne', 'num_cuenca', 'nae', 'naem', 'xetrs89', 'yetrs89', 'cod_saih']

Muestra:


,ref_ceh,nom_embalse,serv,comentario,muni_id,hoja_id,xutm,yutm,xutm30,yutm30,...,longwgs84,latwgs84,mna,mnne,num_cuenca,nae,naem,xetrs89,yetrs89,cod_saih
0,1627,BELESAR,1,LOS DATOS DESDE 1962 EN ADELANTE ESTAN EN PROC...,27016,155,605695,4720600,113659,4730564,...,-74245,423743,330.0,330.0,1408,58,59,113548,4730358,E001
1,1629,"PEARES, LOS",1,RESERVA MODIFICADA DESDE 1955 (REFERENCIA RESP...,27041,188,605020,4702424,111695,4712451,...,-74327,422754,194.2,194.2,1410,63,66,111584,4712244,E002
2,1631,VELLE,1,LOS DATOS DE SALIDAS DEL 10/2000 AL 01/2003 SU...,32054,188,594674,4690377,100534,4701144,...,-75105,422129,108.4,108.4,1467,51,51,100423,4700937,E030
3,1634,CASTRELO,1,LOS DATOS DE SALIDAS DEL 10/2000 AL 01/2003 SU...,32069,225,572962,4682849,78272,4695155,...,-80659,421733,88.0,88.0,1473,51,51,78160,4694948,E031
4,1637,ALBARELLOS,1,LOS DATOS DE SALIDAS DEL 10/2000 AL 01/2003 SO...,32040,186,566619,4694677,72768,4707428,...,-81131,422359,0.0,265.0,1476,48,49,72656,4707221,E032


In [4]:
# Si existe columna de cuenca, ver distribución
if "num_cuenca" in embalses.columns:
    print("=== Embalses por número de cuenca ===")
    print(embalses["num_cuenca"].value_counts().sort_index().head(20))

=== Embalses por número de cuenca ===
num_cuenca
1408    1
1410    1
1414    3
1419    1
1432    1
1434    1
1436    2
1437    2
1438    1
1440    1
1441    1
1448    1
1450    1
1452    2
1453    1
1454    1
1461    2
1462    1
1464    1
1466    1
Name: count, dtype: int64


## 3. Filtrado de embalses Miño-Sil

Para identificar los embalses de nuestra cuenca de estudio, cruzamos por número de cuenca. Los embalses de la cuenca del Miño-Sil tienen números de cuenca que empiezan por `10` (código de la gran cuenca Miño-Sil).

In [5]:
# Filtrar embalses de la gran cuenca Miño-Sil
# num_cuenca es de tipo 10XXX para todas las subcuencas del Miño-Sil
embalses_minosil = embalses #[embalses["num_cuenca"].astype(str).str.startswith(str(GR_CUENCA_MINOSIL))]

print(f"Embalses en la Demarcación Miño-Sil: {len(embalses_minosil)}")
print(f"\nMuestra:")
embalses_minosil[["ref_ceh", "nom_embalse", "num_cuenca", "cod_saih"]].head(10)

Embalses en la Demarcación Miño-Sil: 35

Muestra:


,ref_ceh,nom_embalse,num_cuenca,cod_saih
0,1627,BELESAR,1408,E001
1,1629,"PEARES, LOS",1410,E002
2,1631,VELLE,1467,E030
3,1634,CASTRELO,1473,E031
4,1637,ALBARELLOS,1476,E032
5,1641,FRIEIRA,1491,E033
6,1704,"ROZAS, LAS - MATALAVILLA (SIST.)",1414,E003-E005
7,1709,BARCENA,1414,E007
8,1710,FUENTE DEL AZUFRE,1414,E008
9,1711,CAMPAÑANA,1976,E350


## 4. Cargar y filtrar los datos diarios de embalses (AFLIQE)

La tabla AFLIQE contiene la reserva diaria y la salida diaria de cada embalse. Filtramos por los embalses del Miño-Sil y por el periodo de estudio.

In [6]:
# Cargar datos diarios de embalses
print("Cargando afliqe.csv...")
afliqe = pd.read_csv(DIR_RAW_ANUARIO / "afliqe.csv", sep=";", encoding="latin-1")
print(f"Registros totales: {len(afliqe):,}")
print(f"Columnas: {list(afliqe.columns)}")
afliqe.head()

Cargando afliqe.csv...
Registros totales: 661,369
Columnas: ['ref_ceh', 'fecha', 'reserva', 'salida', 'tipo']


,ref_ceh,fecha,reserva,salida,tipo
0,1627,01/02/1963,429.0,60.3,2
1,1627,02/02/1963,434.0,73.2,2
2,1627,03/02/1963,436.0,82.0,2
3,1627,04/02/1963,436.0,83.4,2
4,1627,05/02/1963,436.0,80.8,2


In [7]:
# Convertir fecha a datetime
afliqe["fecha"] = pd.to_datetime(afliqe["fecha"], format="%d/%m/%Y", errors="coerce")

# Filtrar por embalses Miño-Sil y por periodo
refs_minosil = embalses_minosil["ref_ceh"].tolist()
afliqe_minosil = afliqe[
    (afliqe["ref_ceh"].isin(refs_minosil))
    & (afliqe["fecha"] >= FECHA_INICIO)
    & (afliqe["fecha"] <= FECHA_FIN)
].copy()

print(f"Registros AFLIQE filtrados (Miño-Sil, 2000-2023): {len(afliqe_minosil):,}")
print(f"Embalses distintos con datos: {afliqe_minosil['ref_ceh'].nunique()}")
print(f"Rango de fechas: {afliqe_minosil['fecha'].min()} a {afliqe_minosil['fecha'].max()}")

Registros AFLIQE filtrados (Miño-Sil, 2000-2023): 257,534
Embalses distintos con datos: 34
Rango de fechas: 2000-01-01 00:00:00 a 2020-09-30 00:00:00


## 5. Cargar y filtrar los datos diarios de aforo en río (AFLIQ)

La tabla AFLIQ contiene el caudal medio diario de cada estación de aforo en río. Filtramos análogamente.

In [8]:
# Cargar catálogo de estaciones de aforo
estaf = pd.read_csv(DIR_RAW_ANUARIO / "estaf.csv", sep=";", encoding="latin-1")
print(f"Total estaciones de aforo: {len(estaf)}")

# Filtrar estaciones Miño-Sil
estaf_minosil = estaf[estaf["num_cuenca"].astype(str).str.startswith(str(GR_CUENCA_MINOSIL))]
print(f"Estaciones de aforo en Miño-Sil: {len(estaf_minosil)}")
estaf_minosil[["indroea", "lugar", "num_cuenca", "cod_saih"]].head(10)

Total estaciones de aforo: 90
Estaciones de aforo en Miño-Sil: 63


,indroea,lugar,num_cuenca,cod_saih
0,1010,PUENTE QUEROL,1414,NaN
1,1011,RUAPETIN,1436,NaN
2,1012,VILLAFRANCA DEL BIERZO,1426,NaN
3,1018,PUENTE QUIROGA,1454,NaN
4,1019,PACIOS DE VEIGA,1463,NaN
15,1624,"PARAMO, EL",1401,A015
19,1629,"PEARES, LOS",1410,NaN
20,1631,ORENSE,1471,N010
21,1637,ARENTEIRO,1478,NaN
22,1638,CARBALLINO,1479,A037


In [9]:
# Cargar datos diarios de aforo en río
print("Cargando afliq.csv...")
afliq = pd.read_csv(DIR_RAW_ANUARIO / "afliq.csv", sep=";", encoding="latin-1")
print(f"Registros totales: {len(afliq):,}")

# Convertir fecha y filtrar
afliq["fecha"] = pd.to_datetime(afliq["fecha"], format="%d/%m/%Y", errors="coerce")
indroeas_minosil = estaf_minosil["indroea"].tolist()

afliq_minosil = afliq[
    (afliq["indroea"].isin(indroeas_minosil))
    & (afliq["fecha"] >= FECHA_INICIO)
    & (afliq["fecha"] <= FECHA_FIN)
].copy()

print(f"Registros AFLIQ filtrados (Miño-Sil, 2000-2023): {len(afliq_minosil):,}")
print(f"Estaciones distintas con datos: {afliq_minosil['indroea'].nunique()}")

Cargando afliq.csv...
Registros totales: 697,283
Registros AFLIQ filtrados (Miño-Sil, 2000-2023): 157,952
Estaciones distintas con datos: 37


## 6. Cargar y filtrar los datos mensuales de evaporación (EVAP)

La tabla EVAP contiene la evaporación mensual, útil para calcular el balance hídrico y como variable predictora complementaria.

In [10]:
# Cargar catálogo de estaciones evaporimétricas
estev = pd.read_csv(DIR_RAW_ANUARIO / "estev.csv", sep=";", encoding="latin-1")
print(f"Total estaciones evaporimétricas: {len(estev)}")

# Filtrar por gran cuenca
if "num_cuenca" in estev.columns:
    estev_minosil = estev[estev["num_cuenca"].astype(str).str.startswith(str(GR_CUENCA_MINOSIL))]
else:
    # Alternativa: usar ref_ceh vinculada a los embalses de Miño-Sil
    estev_minosil = estev[estev["ref_ceh"].isin(refs_minosil)]

print(f"Estaciones evaporimétricas en Miño-Sil: {len(estev_minosil)}")
estev_minosil.head()

Total estaciones evaporimétricas: 5
Estaciones evaporimétricas en Miño-Sil: 5


,ref_evap,ref_ceh,lugar,comentario,serv,muni_id,alti,hoja_id,nae,num_cuenca,xutm30,yutm30,longwgs84,latwgs84,xetrs89,yetrs89
0,1002,1709.0,BARCENA,EMBALSE. INT 2007,0,24115,530,158,36,1414,208410,4720885,-63317,423500,208300,4720679
1,1003,1796.0,VILASOUTO,EMBALSE. INT 2007,0,27024,438,156,36,1464,137500,4732825,-72527,423938,137389,4732619
2,1004,NaN,PETIN,INT 1974,0,32060,0,190,5,1436,160825,4700805,-70717,422301,160714,4700599
3,1005,NaN,TRIVES,INT 1974,0,32063,0,189,4,1451,150015,4696490,-71459,422024,149904,4696283
4,1006,1770.0,"PORTAS, LAS",EMBALSE. INT 1974,0,32092,755,0,5,1438,151925,4671155,-71241,420648,151814,4670948


In [11]:
# Cargar datos mensuales de evaporación
evap = pd.read_csv(DIR_RAW_ANUARIO / "evap.csv", sep=";", encoding="latin-1")
print(f"Registros EVAP totales: {len(evap):,}")

# Filtrar por estaciones evap Miño-Sil
ref_evap_minosil = estev_minosil["ref_evap"].tolist()
evap_minosil = evap[evap["ref_evap"].isin(ref_evap_minosil)].copy()

print(f"Registros EVAP filtrados (Miño-Sil): {len(evap_minosil):,}")

Registros EVAP totales: 944
Registros EVAP filtrados (Miño-Sil): 944


## 7. Guardado en `data/interim/`

Guardamos los datos filtrados en formato parquet para su uso posterior en el pipeline.

In [12]:
# Guardar catálogos y datos filtrados
salidas = {
    "anuario_embalses_catalogo.parquet": embalses_minosil,
    "anuario_estaciones_aforo_catalogo.parquet": estaf_minosil,
    "anuario_estaciones_evap_catalogo.parquet": estev_minosil,
    "anuario_afliqe_minosil.parquet": afliqe_minosil,
    "anuario_afliq_minosil.parquet": afliq_minosil,
    "anuario_evap_minosil.parquet": evap_minosil,
}

for nombre, df in salidas.items():
    ruta = DIR_INTERIM / nombre
    df.to_parquet(ruta, index=False)
    print(f"  {nombre}: {len(df):,} filas")

print(f"\nTodo guardado en {DIR_INTERIM}")

  anuario_embalses_catalogo.parquet: 35 filas
  anuario_estaciones_aforo_catalogo.parquet: 63 filas
  anuario_estaciones_evap_catalogo.parquet: 5 filas
  anuario_afliqe_minosil.parquet: 257,534 filas
  anuario_afliq_minosil.parquet: 157,952 filas
  anuario_evap_minosil.parquet: 944 filas

Todo guardado en ..\data\interim


## 8. Resumen final

Balance del filtrado del Anuario:

- **Embalses del Miño-Sil identificados** y sus datos diarios de reserva y salida (2000-2023).
- **Estaciones de aforo del Miño-Sil identificadas** y sus caudales medios diarios.
- **Estaciones evaporimétricas del Miño-Sil identificadas** y sus datos mensuales de evaporación.

Los ficheros quedan disponibles en `data/interim/` para su uso en el resto del pipeline.